# S6 — Detection Power & Metric Ablation Analysis

Uses S3's permutation test results (114K cases) and raw null distributions
to analyze detection power across relationship types, function families,
and metric combinations.

**Sections:**
1. Four-category performance table
2. Mean-only: Function × SNR heatmap
3. Variance-only analysis
4. Mean + Variance analysis
5. Metric ablation study
6. Per-metric unique contribution
7. X distribution influence
8. Error & boundary case analysis
9. Threshold sensitivity

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 150,
                     'figure.facecolor': 'white', 'font.size': 10})

DATA_DIR = Path('generated_scatterplot_data')
S3_DIR   = DATA_DIR / 'full' / 'S3'
VIZ_DIR  = DATA_DIR / 'full' / 'S6_power'
VIZ_DIR.mkdir(parents=True, exist_ok=True)
print(f'Output → {VIZ_DIR}')

In [ ]:
# Load S3 results + case metadata
s3 = pd.read_parquet(S3_DIR / 'permutation_test.parquet')
cases = pd.read_csv(DATA_DIR / 'cases.csv', low_memory=False)
assert len(s3) == len(cases)

# Four-category classification
is_null_func = cases['family_id'] == 'Null'
is_const_spread = cases['spread_pattern'] == 'constant'

cases['category'] = 'mean+variance'
cases.loc[is_null_func & is_const_spread, 'category'] = 'true_null'
cases.loc[~is_null_func & is_const_spread, 'category'] = 'mean_only'
cases.loc[is_null_func & ~is_const_spread, 'category'] = 'variance_only'

s3['category'] = cases['category'].values
s3['family_id'] = cases['family_id'].values
s3['snr'] = cases['snr'].values
s3['spread_pattern'] = cases['spread_pattern'].values
s3['x_distribution'] = cases['x_distribution'].values

# Driver metric
z_cols = ['z_pearson', 'z_spearman', 'z_dcor', 'z_eta2']
s3['driver'] = s3[z_cols].idxmax(axis=1).str.replace('z_', '')

print(f'Total: {len(s3):,} cases')
print(cases['category'].value_counts().to_string())

## 1. Four-Category Performance Table

In [ ]:
cats = ['true_null', 'mean_only', 'variance_only', 'mean+variance']
rows = []
for cat in cats:
    sub = s3[s3['category'] == cat]
    n = len(sub)
    det = (sub['classification'] == 'detectable').sum()
    unc = (sub['classification'] == 'uncertain').sum()
    nd  = (sub['classification'] == 'not_detectable').sum()
    rows.append({
        'Category': cat, 'n': n,
        'Detectable': f'{det} ({det/n:.1%})',
        'Uncertain': f'{unc} ({unc/n:.1%})',
        'Not detectable': f'{nd} ({nd/n:.1%})',
    })

perf_df = pd.DataFrame(rows)
print('=== Four-Category Performance ===')
print(perf_df.to_string(index=False))
print()
print('Note: True Null "detectable" = false positives.')
print('      Variance-only "detectable" = correct (real dependence in variance).')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
cat_labels = ['True Null', 'Mean-only', 'Variance-only', 'Mean+Variance']
det_rates = []
for cat in cats:
    sub = s3[s3['category'] == cat]
    det_rates.append((sub['classification'] == 'detectable').mean())

colors = ['#ef4444', '#3b82f6', '#f59e0b', '#10b981']
bars = ax.barh(range(4), det_rates, color=colors, alpha=0.7)
ax.set_yticks(range(4))
ax.set_yticklabels(cat_labels)
ax.set_xlabel('Detection rate')
ax.set_title('Detection Rate by Relationship Category')
ax.axvline(0.05, color='red', ls='--', lw=1, alpha=0.5, label='α=0.05')
for i, (rate, cat) in enumerate(zip(det_rates, cats)):
    n = (s3['category'] == cat).sum()
    ax.text(rate + 0.01, i, f'{rate:.1%} (n={n:,})', va='center', fontsize=9)
ax.set_xlim(0, 1.15)
ax.legend()
plt.tight_layout()
fig.savefig(VIZ_DIR / '1_four_category_performance.png', bbox_inches='tight')
plt.show()

## 2. Mean-only Detection Power

f(x) ≠ constant, σ(x) = constant.
This is the cleanest test of mean-signal detection.

In [ ]:
mo = s3[s3['category'] == 'mean_only'].copy()
mo['snr_num'] = pd.to_numeric(mo['snr'], errors='coerce')

# Function × SNR detection rate heatmap
funcs = sorted(mo['family_id'].unique())
snrs = sorted(mo['snr_num'].dropna().unique())

heat = np.full((len(funcs), len(snrs)), np.nan)
for i, f in enumerate(funcs):
    for j, s in enumerate(snrs):
        sub = mo[(mo['family_id'] == f) & (mo['snr_num'] == s)]
        if len(sub) > 0:
            heat[i, j] = (sub['classification'] == 'detectable').mean()

fig, ax = plt.subplots(figsize=(14, 8))
im = ax.imshow(heat, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
ax.set_xticks(range(len(snrs)))
ax.set_xticklabels([f'{s:.1f}' if s < 100 else ('∞' if np.isinf(s) else f'{s:.0f}') for s in snrs],
                   rotation=45, fontsize=9)
ax.set_yticks(range(len(funcs)))
ax.set_yticklabels(funcs, fontsize=9)
ax.set_xlabel('SNR')
ax.set_ylabel('Function Family')
ax.set_title('Mean-only Detection Rate: Function × SNR')

for i in range(len(funcs)):
    for j in range(len(snrs)):
        if not np.isnan(heat[i, j]):
            color = 'white' if heat[i, j] < 0.5 else 'black'
            ax.text(j, i, f'{heat[i,j]:.0%}', ha='center', va='center',
                    fontsize=7, color=color)

plt.colorbar(im, ax=ax, label='Detection rate', shrink=0.8)
plt.tight_layout()
fig.savefig(VIZ_DIR / '2_mean_only_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
# Detection rate vs SNR curve (mean-only, aggregated)
fig, ax = plt.subplots(figsize=(10, 5))

for f in funcs:
    rates = []
    for s in snrs:
        sub = mo[(mo['family_id'] == f) & (mo['snr_num'] == s)]
        rates.append((sub['classification'] == 'detectable').mean() if len(sub) > 0 else np.nan)
    ax.plot(range(len(snrs)), rates, 'o-', alpha=0.4, markersize=3, label=f)

# Overall mean-only curve
overall = [mo[mo['snr_num'] == s]['classification'].eq('detectable').mean() for s in snrs]
ax.plot(range(len(snrs)), overall, 'k-', lw=3, marker='s', markersize=6, label='Overall', zorder=10)

ax.set_xticks(range(len(snrs)))
ax.set_xticklabels([f'{s:.1f}' if s < 100 else ('∞' if np.isinf(s) else f'{s:.0f}') for s in snrs],
                   rotation=45)
ax.set_xlabel('SNR')
ax.set_ylabel('Detection rate')
ax.set_title('Mean-only Detection Rate vs SNR (each line = one function family)')
ax.set_ylim(-0.05, 1.05)
ax.axhline(0.95, color='gray', ls=':', lw=1, alpha=0.5)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=7, ncol=2)
plt.tight_layout()
fig.savefig(VIZ_DIR / '3_mean_only_snr_curves.png', bbox_inches='tight')
plt.show()

In [ ]:
# Which metric drives detection in mean-only cases?
mo_det = mo[mo['classification'] == 'detectable']
print('Mean-only: driver metric distribution')
print(mo_det['driver'].value_counts().to_string())
print()

# Driver by SNR
fig, ax = plt.subplots(figsize=(12, 5))
driver_names = ['eta2', 'dcor', 'pearson', 'spearman']
driver_colors = {'eta2': '#f59e0b', 'dcor': '#10b981', 'pearson': '#3b82f6', 'spearman': '#ef4444'}
bottom = np.zeros(len(snrs))
for drv in driver_names:
    fracs = []
    for s in snrs:
        sub = mo_det[mo_det['snr_num'] == s]
        fracs.append((sub['driver'] == drv).mean() if len(sub) > 0 else 0)
    ax.bar(range(len(snrs)), fracs, bottom=bottom, label=drv,
           color=driver_colors[drv], alpha=0.7)
    bottom += fracs

ax.set_xticks(range(len(snrs)))
ax.set_xticklabels([f'{s:.1f}' if s < 100 else ('∞' if np.isinf(s) else f'{s:.0f}') for s in snrs],
                   rotation=45)
ax.set_xlabel('SNR')
ax.set_ylabel('Fraction')
ax.set_title('Mean-only: Which Metric Drives Detection? (by SNR)')
ax.legend()
plt.tight_layout()
fig.savefig(VIZ_DIR / '4_mean_only_driver_by_snr.png', bbox_inches='tight')
plt.show()

## 3. Variance-only Analysis

f(x) = 0, σ(x) ≠ constant. Only noise spread depends on x.

In [ ]:
vo = s3[s3['category'] == 'variance_only'].copy()
print(f'Variance-only cases: {len(vo)}')
print()

# Detection by spread pattern
print('Detection by spread_pattern:')
for sp in ['increasing', 'decreasing', 'middle_high']:
    sub = vo[vo['spread_pattern'] == sp]
    det = (sub['classification'] == 'detectable').mean()
    print(f'  {sp:15s}: {det:.1%} (n={len(sub)})')
print()

# Which metric drives?
vo_det = vo[vo['classification'] == 'detectable']
print('Driver metric:')
print(vo_det['driver'].value_counts().to_string())
print()

# Per-metric Z-scores for variance-only
print('Mean Z-scores (variance-only cases):')
for zc in z_cols:
    print(f'  {zc:15s}: {vo[zc].mean():.2f} ± {vo[zc].std():.2f}')

In [ ]:
# Z-score comparison: variance-only vs true-null
tn = s3[s3['category'] == 'true_null']

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for i, zc in enumerate(z_cols):
    ax = axes[i]
    ax.hist(tn[zc], bins=40, alpha=0.5, label='True Null', density=True, color='steelblue')
    ax.hist(vo[zc], bins=40, alpha=0.5, label='Variance-only', density=True, color='salmon')
    ax.set_title(zc.replace('z_', ''))
    ax.legend(fontsize=8)
    ax.set_xlabel('Z-score')

plt.suptitle('Z-score Distributions: True Null vs Variance-only', fontsize=13)
plt.tight_layout()
fig.savefig(VIZ_DIR / '5_variance_only_zscore.png', bbox_inches='tight')
plt.show()

## 4. Mean + Variance Analysis

f(x) ≠ 0, σ(x) ≠ constant. Both mean and noise spread depend on x.

In [ ]:
mv = s3[s3['category'] == 'mean+variance'].copy()
mv['snr_num'] = pd.to_numeric(mv['snr'], errors='coerce')

# Compare detection rate: mean_only vs mean+variance at same SNR
fig, ax = plt.subplots(figsize=(10, 5))
for cat, label, color in [('mean_only', 'Mean-only', '#3b82f6'),
                           ('mean+variance', 'Mean+Variance', '#10b981')]:
    sub = s3[s3['category'] == cat].copy()
    sub['snr_num'] = pd.to_numeric(sub['snr'], errors='coerce')
    rates = [sub[sub['snr_num']==s]['classification'].eq('detectable').mean() for s in snrs]
    ax.plot(range(len(snrs)), rates, 'o-', lw=2, label=label, color=color)

ax.set_xticks(range(len(snrs)))
ax.set_xticklabels([f'{s:.1f}' if s < 100 else ('∞' if np.isinf(s) else f'{s:.0f}') for s in snrs],
                   rotation=45)
ax.set_xlabel('SNR')
ax.set_ylabel('Detection rate')
ax.set_title('Detection Rate: Mean-only vs Mean+Variance')
ax.legend()
ax.set_ylim(0.9, 1.005)
plt.tight_layout()
fig.savefig(VIZ_DIR / '6_mean_vs_meanvar.png', bbox_inches='tight')
plt.show()

# At low SNR, what drives detection in mean+variance?
mv_low = mv[mv['snr_num'] <= 0.5]
mv_low_det = mv_low[mv_low['classification'] == 'detectable']
print(f'Mean+Variance at SNR ≤ 0.5: {len(mv_low_det)}/{len(mv_low)} detectable')
print('Driver metric:')
print(mv_low_det['driver'].value_counts().to_string())

## 5. Metric Ablation Study

Re-construct joint tests using subsets of metrics, without rerunning permutations.
Uses the raw NPZ null arrays to recompute T and p-values.

In [ ]:
# Load raw null arrays
print('Loading raw null distributions (this may take a minute)...')
p1 = np.load(S3_DIR / '_perm_phase1.npz')
p2 = np.load(S3_DIR / '_perm_phase2.npz')

obs_arrays = {
    'pearson':  p1['pearson_obs'],
    'spearman': p1['spearman_obs'],
    'eta2':     p1['eta2_obs'],
    'dcor':     p2['dcor_obs'],
}
null_arrays = {
    'pearson':  p1['pearson_null'],
    'spearman': p1['spearman_null'],
    'eta2':     p1['eta2_null'],
    'dcor':     p2['dcor_null'],
}

n_cases = len(obs_arrays['pearson'])
n_perm = null_arrays['pearson'].shape[1]
print(f'Loaded: {n_cases:,} cases × {n_perm} permutations')

In [ ]:
def run_ablation(metric_subset, obs_arrays, null_arrays, n_cases, n_perm):
    """Recompute joint test using only the given metric subset."""
    # Compute Z-scores for all cases
    z_obs_all = {}   # metric -> (n_cases,)
    z_null_all = {}  # metric -> (n_cases, n_perm)

    for m in metric_subset:
        obs = obs_arrays[m]
        null = null_arrays[m].astype(np.float64)
        med = np.median(null, axis=1)
        q75 = np.percentile(null, 75, axis=1)
        q25 = np.percentile(null, 25, axis=1)
        iqr = q75 - q25
        # Fallback
        tiny = iqr < 1e-12
        iqr[tiny] = np.std(null[tiny], axis=1) * 1.35
        still_tiny = iqr < 1e-12
        iqr[still_tiny] = 1.0  # avoid div by zero

        z_obs_all[m] = (obs - med) / iqr
        z_null_all[m] = (null - med[:, None]) / iqr[:, None]

    # Joint test: T = max(Z) over subset
    z_obs_stack = np.stack([z_obs_all[m] for m in metric_subset])  # (len(subset), n_cases)
    z_null_stack = np.stack([z_null_all[m] for m in metric_subset])  # (len(subset), n_cases, n_perm)

    T_obs = z_obs_stack.max(axis=0)  # (n_cases,)
    T_null = z_null_stack.max(axis=0)  # (n_cases, n_perm)

    p_values = (np.sum(T_null >= T_obs[:, None], axis=1) + 1) / (n_perm + 1)
    classification = np.where(p_values <= 0.05, 'detectable',
                     np.where(p_values >= 0.10, 'not_detectable', 'uncertain'))
    return p_values, classification

print('Ablation function ready')

In [ ]:
# Define metric subsets to test
subsets = {
    'pearson only':            ['pearson'],
    'spearman only':           ['spearman'],
    'dcor only':               ['dcor'],
    'eta2 only':               ['eta2'],
    'pearson+spearman':        ['pearson', 'spearman'],
    'pearson+spearman+eta2':   ['pearson', 'spearman', 'eta2'],
    'pearson+spearman+dcor':   ['pearson', 'spearman', 'dcor'],
    'all four':                ['pearson', 'spearman', 'dcor', 'eta2'],
    'no dcor':                 ['pearson', 'spearman', 'eta2'],
    'no eta2':                 ['pearson', 'spearman', 'dcor'],
    'no pearson':              ['spearman', 'dcor', 'eta2'],
    'no spearman':             ['pearson', 'dcor', 'eta2'],
}

cat_arr = cases['category'].values

ablation_rows = []
for name, mlist in subsets.items():
    print(f'  Running: {name} ...')
    pv, clf = run_ablation(mlist, obs_arrays, null_arrays, n_cases, n_perm)
    row = {'Metrics': name}
    for cat in cats:
        mask = cat_arr == cat
        det = (clf[mask] == 'detectable').mean()
        row[cat] = det
    ablation_rows.append(row)

abl_df = pd.DataFrame(ablation_rows)
print('\nDone!')

In [ ]:
# Format ablation results
print('=== Metric Ablation Results ===')
print(f'{"Metrics":>25s}  {"True Null FPR":>14s}  {"Mean-only":>10s}  {"Var-only":>10s}  {"Mean+Var":>10s}')
print('-' * 75)
for _, r in abl_df.iterrows():
    tn_flag = ' !' if r['true_null'] > 0.07 else '  '
    print(f'{r["Metrics"]:>25s}  {r["true_null"]:>12.1%}{tn_flag}  {r["mean_only"]:>9.1%}  {r["variance_only"]:>9.1%}  {r["mean+variance"]:>9.1%}')
print()
print('! = FPR > 7% (potential calibration concern)')

In [ ]:
# Visualize ablation as grouped bar chart
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, cat, title in zip(axes, ['mean_only', 'variance_only', 'mean+variance'],
                          ['Mean-only Power', 'Variance-only Power', 'Mean+Var Power']):
    vals = abl_df[cat].values
    names_short = [r['Metrics'] for _, r in abl_df.iterrows()]
    y_pos = range(len(names_short))
    colors = ['#ef4444' if 'only' in n else '#3b82f6' if 'no ' in n else '#10b981' for n in names_short]
    ax.barh(y_pos, vals, color=colors, alpha=0.7)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(names_short, fontsize=8)
    ax.set_xlabel('Detection rate')
    ax.set_title(title)
    ax.set_xlim(0, 1.05)
    for i, v in enumerate(vals):
        ax.text(v + 0.01, i, f'{v:.1%}', va='center', fontsize=7)

plt.suptitle('Metric Ablation: Detection Power by Category', fontsize=13)
plt.tight_layout()
fig.savefig(VIZ_DIR / '7_ablation_power.png', bbox_inches='tight')
plt.show()

## 6. Per-Metric Unique Contribution

U_m = P(full test detects, but test without m does not)

In [ ]:
# Compute unique contributions
# Full test classification
_, clf_full = run_ablation(['pearson', 'spearman', 'dcor', 'eta2'],
                           obs_arrays, null_arrays, n_cases, n_perm)
full_det = clf_full == 'detectable'

leave_out = {
    'pearson':  ['spearman', 'dcor', 'eta2'],
    'spearman': ['pearson', 'dcor', 'eta2'],
    'dcor':     ['pearson', 'spearman', 'eta2'],
    'eta2':     ['pearson', 'spearman', 'dcor'],
}

print('=== Unique Contribution per Metric ===')
print(f'{"Metric":>10s}  {"Mean-only":>10s}  {"Var-only":>10s}  {"Mean+Var":>10s}  {"Overall":>10s}')
print('-' * 55)

for metric, remaining in leave_out.items():
    _, clf_without = run_ablation(remaining, obs_arrays, null_arrays, n_cases, n_perm)
    without_det = clf_without == 'detectable'
    unique = full_det & ~without_det  # detected by full, not by leave-one-out

    parts = []
    for cat in ['mean_only', 'variance_only', 'mean+variance']:
        mask = cat_arr == cat
        n_cat = mask.sum()
        u = unique[mask].sum()
        parts.append(f'{u/n_cat:.2%}')
    overall = unique.sum() / n_cases
    print(f'{metric:>10s}  {parts[0]:>10s}  {parts[1]:>10s}  {parts[2]:>10s}  {overall:>9.2%}')

## 7. X Distribution Influence

In [ ]:
# Detection rate by x_distribution, relative to 'even'
mo_xdist = mo.copy()
xdists = sorted(mo_xdist['x_distribution'].unique())

even_rate = (mo_xdist[mo_xdist['x_distribution'] == 'even']['classification'] == 'detectable').mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Absolute rates
ax = axes[0]
rates_xd = []
for xd in xdists:
    sub = mo_xdist[mo_xdist['x_distribution'] == xd]
    rates_xd.append((sub['classification'] == 'detectable').mean())
ax.barh(range(len(xdists)), rates_xd, color='steelblue', alpha=0.7)
ax.set_yticks(range(len(xdists)))
ax.set_yticklabels(xdists)
ax.set_xlabel('Detection rate')
ax.set_title('Mean-only: Detection Rate by x_distribution')

# Relative to even
ax = axes[1]
deltas = [r - even_rate for r in rates_xd]
colors = ['salmon' if d < -0.005 else 'lightgreen' for d in deltas]
ax.barh(range(len(xdists)), deltas, color=colors, alpha=0.7)
ax.set_yticks(range(len(xdists)))
ax.set_yticklabels(xdists)
ax.set_xlabel('ΔP (relative to even)')
ax.set_title('Detection Rate Change vs even')
ax.axvline(0, color='k', lw=0.5)

plt.tight_layout()
fig.savefig(VIZ_DIR / '8_xdist_influence.png', bbox_inches='tight')
plt.show()

In [ ]:
# X distribution effect at low SNR only
mo_low = mo[mo['snr_num'] <= 1.0]
print('Mean-only detection rate at low SNR (≤1.0) by x_distribution:')
for xd in xdists:
    sub = mo_low[mo_low['x_distribution'] == xd]
    det = (sub['classification'] == 'detectable').mean()
    print(f'  {xd:>15s}: {det:.1%} (n={len(sub)})')

## 8. Error & Boundary Case Analysis

In [ ]:
# False negatives: signal cases classified as not_detectable
signal = s3[s3['category'].isin(['mean_only', 'mean+variance'])]
fn = signal[signal['classification'] == 'not_detectable'].copy()
fn['snr_num'] = pd.to_numeric(fn['snr'], errors='coerce')
print(f'False negatives (signal, not_detectable): {len(fn)}')
print()
print('By category:')
print(fn['category'].value_counts().to_string())
print()
print('By SNR:')
print(fn['snr_num'].value_counts().sort_index().to_string())
print()
print('By function family:')
print(fn['family_id'].value_counts().head(10).to_string())
print()
print('By x_distribution:')
print(fn['x_distribution'].value_counts().to_string())
print()
print('By spread_pattern:')
print(fn['spread_pattern'].value_counts().to_string())

In [ ]:
# Uncertain cases
unc = s3[s3['classification'] == 'uncertain'].copy()
unc['snr_num'] = pd.to_numeric(unc['snr'], errors='coerce')
print(f'Uncertain cases: {len(unc)}')
print()
print('By category:')
print(unc['category'].value_counts().to_string())
print()
print('By SNR (signal cases):')
print(unc[unc['category'] != 'true_null']['snr_num'].value_counts().sort_index().to_string())

## 9. Threshold Sensitivity

How do FPR and detection power change with different α thresholds?

In [ ]:
alphas = [0.001, 0.005, 0.01, 0.02, 0.05, 0.10, 0.15, 0.20]
pv = s3['p_value'].values

print(f'{"α":>6s}  {"True Null FPR":>14s}  {"Mean-only":>10s}  {"Var-only":>10s}  {"Mean+Var":>10s}')
print('-' * 60)
for alpha in alphas:
    det = pv <= alpha
    parts = []
    for cat in cats:
        mask = cat_arr == cat
        parts.append(det[mask].mean())
    print(f'{alpha:>6.3f}  {parts[0]:>13.2%}  {parts[1]:>9.1%}  {parts[2]:>9.1%}  {parts[3]:>9.1%}')

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
for cat, label, color in [('true_null', 'True Null FPR', '#ef4444'),
                           ('mean_only', 'Mean-only power', '#3b82f6'),
                           ('variance_only', 'Var-only power', '#f59e0b'),
                           ('mean+variance', 'Mean+Var power', '#10b981')]:
    mask = cat_arr == cat
    rates = [pv[mask].le(a).mean() if hasattr(pv[mask], 'le') else (pv[mask] <= a).mean() for a in alphas]
    ax.plot(alphas, rates, 'o-', label=label, color=color, lw=2)

ax.set_xlabel('α threshold')
ax.set_ylabel('Rate')
ax.set_title('FPR and Detection Power vs α Threshold')
ax.legend()
ax.set_xscale('log')
plt.tight_layout()
fig.savefig(VIZ_DIR / '9_threshold_sensitivity.png', bbox_inches='tight')
plt.show()

## 10. Summary

In [ ]:
print('='*60)
print('S6 ANALYSIS SUMMARY')
print('='*60)
print()
print('1. Four-Category Performance:')
for _, r in perf_df.iterrows():
    print(f'   {r["Category"]:>18s}: {r["Detectable"]}')
print()
print('2. Metric Ablation (key findings):')
for _, r in abl_df.iterrows():
    print(f'   {r["Metrics"]:>25s}: mean-only={r["mean_only"]:.1%}, var-only={r["variance_only"]:.1%}')
print()
print('3. False negatives concentrated at low SNR and specific functions.')
print('4. Method is robust across x_distributions.')